# PartitionNet (v28-eq-vector)
### Taxon-Permutation-Equivariant Neural Architecture for Phylogenetic Quartet Partitioning

---

## 📌 1. Overview & Mathematical Formulation
Given an unrooted quartet tree with 4 taxa $\mathcal{T} = \{t_1, t_2, t_3, t_4\}$, there are 3 possible unrooted topologies:
1. **Topology 1**: $(t_1, t_2) \mid (t_3, t_4)$
2. **Topology 2**: $(t_1, t_3) \mid (t_2, t_4)$
3. **Topology 3**: $(t_1, t_4) \mid (t_2, t_3)$

Each topology corresponds to a **bipolar partition vector** $y \in \{-1, +1\}^4$ defined up to a global sign flip ($y \equiv -y$):
- Topo 1: $[+1, +1, -1, -1] \equiv [-1, -1, +1, +1]$
- Topo 2: $[+1, -1, +1, -1] \equiv [-1, +1, -1, +1]$
- Topo 3: $[+1, -1, -1, +1] \equiv [-1, +1, +1, -1]$

### 🔒 Permutation Equivariance Guarantee
For any permutation $\pi \in S_4$ of taxa order, **PartitionNet** guarantees:
$$\mathbf{z}(\pi(\mathbf{x})) = \pi(\mathbf{z}(\mathbf{x}))$$
The prediction is strictly invariant to arbitrary ordering of taxa in the input alignment.


In [ ]:
# 🛠️ Installation & Imports
import os
import sys
import time
import math
import random
import h5py
import json
import csv
import gc
import psutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler

# Ensure PartitionNet module is importable
if os.path.exists("./PartitionNet"):
    sys.path.insert(0, "./PartitionNet")
elif os.path.exists("../PartitionNet"):
    sys.path.insert(0, "../PartitionNet")

from partitionnet import (
    PartitionNetEq,
    StableSymmetricBipolarLoss,
    NumericRAMDataset,
    TrainChunkBasedDynamicSampler,
    ChunkCollator,
    ModelConfig,
    TrainingConfig
)
from partitionnet.utils.memory import set_seed, trim_memory, get_ram_usage, clean_state_dict
from partitionnet.utils.metrics import compute_metrics, decode_topos
from partitionnet.utils.invariance import check_permutation_invariance, run_permutation_test

print("PartitionNet imported successfully!")


## ⚙️ 2. Configuration & Paths


In [ ]:
SEED = 9999
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

cfg = TrainingConfig()
WINDOW_SIZE = cfg.window_size
MAX_CHUNKS_LIMIT = cfg.max_chunks_limit
CHUNK_BUCKETS = cfg.chunk_buckets
MAX_BATCH_SIZE_TRAIN = cfg.max_batch_size_train
MAX_CHUNKS_PER_BATCH_TRAIN = cfg.max_chunks_per_batch_train
MAX_BATCH_SIZE_EVAL = cfg.max_batch_size_eval
MAX_CHUNKS_PER_BATCH_EVAL = cfg.max_chunks_per_batch_eval
ACCUM_STEPS = cfg.accum_steps
MAX_EPOCHS = cfg.max_epochs
PATIENCE = cfg.patience
MAX_LR = cfg.max_lr

# Auto-detect data path on Kaggle or local
BASE_H5_PATH = cfg.base_h5_path
if not os.path.exists(BASE_H5_PATH):
    for root, _, files in os.walk("/kaggle/input"):
        if any(f.endswith(".h5") for f in files):
            BASE_H5_PATH = root
            break
print(f"Base H5 Path: {BASE_H5_PATH}")


## 🔬 3. Verify Permutation Equivariance
Before training, we test `PartitionNetEq` on random synthetic quartet alignments under all $4! = 24$ permutations.


In [ ]:
# Permutation Equivariance Check
test_model = PartitionNetEq(embed_dim=16, dim=32, window=2048).to(DEVICE)
test_model.eval()

# Synthetic input: Batch=2, Chunks=2, Taxa=4, Length=2048
dummy_input = torch.randint(0, 4, (2, 2, 4, 2048), dtype=torch.uint8, device=DEVICE)
max_logit_diff, topo_agreement = check_permutation_invariance(test_model, dummy_input)

print(f"Max Logit Difference under Permutations: {max_logit_diff:.3e}")
print(f"Topology Agreement across all 24 Permutations: {topo_agreement * 100:.2f}%")
assert topo_agreement == 1.0, "Permutation equivariance test failed!


## 🚀 4. Model Training Pipeline
The training pipeline features:
- **ChunkCollator & DynamicSampler**: Preallocated memory buffers with dynamic length sorting.
- **StableSymmetricBipolarLoss**: Flip-invariant margin loss + tanh sum constraint + auxiliary pair loss.
- **OneCycleLR Scheduler & AMP Autocast**: Fast convergence and FP16 efficiency.


In [ ]:
# Instantiate Data Collators
train_collator = ChunkCollator(MAX_BATCH_SIZE_TRAIN, MAX_CHUNKS_LIMIT, WINDOW_SIZE, CHUNK_BUCKETS)
eval_collator = ChunkCollator(MAX_BATCH_SIZE_EVAL, MAX_CHUNKS_LIMIT, WINDOW_SIZE, CHUNK_BUCKETS)

# Model, Loss, Optimizer & Scheduler
model = PartitionNetEq(window=WINDOW_SIZE).to(DEVICE)
criterion = StableSymmetricBipolarLoss(lambda_penalty=0.5, temperature=3.0, lambda_pair=0.5)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=MAX_LR, weight_decay=1e-3)
scaler = GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model trainable parameters: {total_params:,}")


## 📊 5. Evaluation & 24-Taxon Permutation Benchmark


In [ ]:
# Test evaluation helper
def run_evaluation(model, loader, device):
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            with autocast('cuda', enabled=(device.type == 'cuda')):
                out = model(inputs)
            logits_list.append(out.cpu().float().numpy())
            labels_list.append(labels.numpy())
    logits = np.concatenate(logits_list, axis=0)
    targets = np.concatenate(labels_list, axis=0)
    return compute_metrics(logits, targets)

print("Evaluation pipeline initialized.")
